### 0) Imports

In [ ]:
from DNN import DNN
from DBN import DBN
from RBM import RBM
import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision import datasets

### 1) Préparation des données

In [ ]:
# similar to transforms.PILToTensor() but binarizes the image
from torchvision.utils import  _log_api_usage_once
from torchvision.transforms import functional as F

class PILToBinTensor:
    """Convert a PIL Image to a tensor of the same type - this does not scale values.

    This transform does not support torchscript.

    Converts a PIL Image (H x W x C) to a Tensor of shape (C x H x W).
    """

    def __init__(self) -> None:
        _log_api_usage_once(self)

    def __call__(self, pic):
        """
        .. note::

            A deep copy of the underlying array is performed.

        Args:
            pic (PIL Image): Image to be converted to tensor.

        Returns:
            Tensor: Converted image.
        """
        tensor = F.pil_to_tensor(pic)
        tensor[tensor <= 127] = 0
        tensor[tensor > 127] = 1
        return tensor

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}()"

In [ ]:
batch_size = 100

In [ ]:
# MNIST Dataset from yann.lecun.com
mnist_trainset = datasets.MNIST(root='./data/', train=True, transform=PILToBinTensor(), download=True)
mnist_testset = datasets.MNIST(root='./data/', train=False, transform=PILToBinTensor(), download=False)

#create data loader
max_mnist_size = 60000
mnist_trainset_reduced = torch.utils.data.random_split(mnist_trainset, [max_mnist_size, len(mnist_trainset)-max_mnist_size])[0] 
mnist_train_loader = torch.utils.data.DataLoader(mnist_trainset_reduced, batch_size=batch_size, shuffle=True,drop_last=True)

# download test dataset
max_mnist_size = 10000
mnist_testset_reduced = torch.utils.data.random_split(mnist_testset, [max_mnist_size, len(mnist_testset)-max_mnist_size])[0] 
mnist_test_loader = torch.utils.data.DataLoader(mnist_testset_reduced, batch_size=batch_size, shuffle=True,drop_last=True)

In [ ]:
n_rows = mnist_trainset_reduced.dataset.data.shape[1]
n_cols = mnist_trainset_reduced.dataset.data.shape[2]
n_channels = 1
n_pixels = n_rows*n_cols
classification_size = 10
img_shape = (n_rows, n_cols, n_channels)

In [ ]:
# Unpack training data and labels
train_data, train_labels, train_clear_label = [], [], []
for data, label in mnist_train_loader:
    train_data.append(data.reshape(-1, n_rows * n_cols * n_channels))
    train_labels.append(torch.nn.functional.one_hot(label, num_classes=classification_size))
    train_clear_label.append(label)
train_data = torch.cat(train_data).numpy()
train_labels = torch.cat(train_labels).numpy()
train_clear_label = torch.cat(train_clear_label).numpy()

# Unpack test data and labels
test_data, test_labels, test_clear_label = [], [], []
for data, label in mnist_test_loader:
    test_data.append(data.reshape(-1, n_rows * n_cols * n_channels))
    test_labels.append(torch.nn.functional.one_hot(label, num_classes=classification_size))
    test_clear_label.append(label)
test_data = torch.cat(test_data).numpy()
test_labels = torch.cat(test_labels).numpy()
test_clear_label = torch.cat(test_clear_label).numpy()

### 2) Configuration d'entraînement

In [ ]:
num_iterations_mnist = 200
num_iterations = 100
learning_rate = 0.1

### 3) Expérimentations
#### 3.1 Variation du nombre de couches

In [ ]:
n_couches = [1, 2, 3, 4, 5, 6, 7]
dnn_pretrain_train_error = []
dnn_pretrain_test_error = []
dnn_random_train_error = []
dnn_random_test_error = []

for n_hidden_layers in n_couches:
    couches_dnn = [n_pixels] + [200] * n_hidden_layers

    dnn_pretrain = DNN()
    dnn_pretrain.init_DNN(couches_dnn, classification_size)
    dnn_pretrain.pretrain_DNN(train_data, num_iterations, learning_rate, batch_size)
    dnn_pretrain.retropropagation(num_iterations_mnist, learning_rate, batch_size, train_data, train_labels)
    dnn_pretrain_train_error.append(dnn_pretrain.test_DNN(train_data, train_clear_label))
    dnn_pretrain_test_error.append(dnn_pretrain.test_DNN(test_data, test_clear_label))

    dnn_random = DNN()
    dnn_random.init_DNN(couches_dnn, classification_size)
    dnn_random.retropropagation(num_iterations_mnist, learning_rate, batch_size, train_data, train_labels)
    dnn_random_train_error.append(dnn_random.test_DNN(train_data, train_clear_label))
    dnn_random_test_error.append(dnn_random.test_DNN(test_data, test_clear_label))

##### Visualisation des résultats (3.1)

In [ ]:
plt.plot(n_couches, dnn_pretrain_test_error, label="DNN pré-entraîné (test)")
plt.scatter(n_couches, dnn_pretrain_test_error)
plt.plot(n_couches, dnn_random_test_error, label="DNN non pré-entraîné (test)")
plt.scatter(n_couches, dnn_random_test_error)
plt.xlabel("nombre de couches")
plt.ylabel("taux d'erreur")
plt.legend()
plt.show()

print("Erreurs train (pré-entraîné):", np.round(dnn_pretrain_train_error, 4))
print("Erreurs test  (pré-entraîné):", np.round(dnn_pretrain_test_error, 4))
print("Erreurs train (aléatoire):", np.round(dnn_random_train_error, 4))
print("Erreurs test  (aléatoire):", np.round(dnn_random_test_error, 4))

#### 3.2 Variation du nombre de neurones par couche

In [ ]:
n_neurones = [100, 300, 500, 700]
dnn_pretrain_train_error = []
dnn_pretrain_test_error = []
dnn_random_train_error = []
dnn_random_test_error = []

for n_neurone in n_neurones:
    couches_dnn = [n_pixels, n_neurone, n_neurone]

    dnn_pretrain = DNN()
    dnn_pretrain.init_DNN(couches_dnn, classification_size)
    dnn_pretrain.pretrain_DNN(train_data, num_iterations, learning_rate, batch_size)
    dnn_pretrain.retropropagation(num_iterations_mnist, learning_rate, batch_size, train_data, train_labels)
    dnn_pretrain_train_error.append(dnn_pretrain.test_DNN(train_data, train_clear_label))
    dnn_pretrain_test_error.append(dnn_pretrain.test_DNN(test_data, test_clear_label))

    dnn_random = DNN()
    dnn_random.init_DNN(couches_dnn, classification_size)
    dnn_random.retropropagation(num_iterations_mnist, learning_rate, batch_size, train_data, train_labels)
    dnn_random_train_error.append(dnn_random.test_DNN(train_data, train_clear_label))
    dnn_random_test_error.append(dnn_random.test_DNN(test_data, test_clear_label))

##### Visualisation des résultats (3.2)

In [ ]:
plt.plot(n_neurones, dnn_pretrain_test_error, label="DNN pré-entraîné (test)")
plt.scatter(n_neurones, dnn_pretrain_test_error)
plt.plot(n_neurones, dnn_random_test_error, label="DNN non pré-entraîné (test)")
plt.scatter(n_neurones, dnn_random_test_error)
plt.xlabel("nombre de neurones")
plt.ylabel("taux d'erreur")
plt.legend()
plt.show()

print("Erreurs train (pré-entraîné):", np.round(dnn_pretrain_train_error, 4))
print("Erreurs test  (pré-entraîné):", np.round(dnn_pretrain_test_error, 4))
print("Erreurs train (aléatoire):", np.round(dnn_random_train_error, 4))
print("Erreurs test  (aléatoire):", np.round(dnn_random_test_error, 4))

#### 3.3 Variation du nombre de données d'apprentissage

In [ ]:
n_donnees_entrainement = [200, 1000, 3000, 7000, 10000, 30000, 60000]
hidden_units = 200
dnn_pretrain_train_error = []
dnn_pretrain_test_error = []
dnn_random_train_error = []
dnn_random_test_error = []

for n_entrainement in n_donnees_entrainement:
    couches_dnn = [n_pixels, hidden_units, hidden_units]

    dnn_pretrain = DNN()
    dnn_pretrain.init_DNN(couches_dnn, classification_size)
    dnn_pretrain.pretrain_DNN(train_data[:n_entrainement], num_iterations, learning_rate, batch_size)
    dnn_pretrain.retropropagation(num_iterations_mnist, learning_rate, batch_size, train_data[:n_entrainement], train_labels[:n_entrainement])
    dnn_pretrain_train_error.append(dnn_pretrain.test_DNN(train_data[:n_entrainement], train_clear_label[:n_entrainement]))
    dnn_pretrain_test_error.append(dnn_pretrain.test_DNN(test_data, test_clear_label))

    dnn_random = DNN()
    dnn_random.init_DNN(couches_dnn, classification_size)
    dnn_random.retropropagation(num_iterations_mnist, learning_rate, batch_size, train_data[:n_entrainement], train_labels[:n_entrainement])
    dnn_random_train_error.append(dnn_random.test_DNN(train_data[:n_entrainement], train_clear_label[:n_entrainement]))
    dnn_random_test_error.append(dnn_random.test_DNN(test_data, test_clear_label))

##### Visualisation des résultats (3.3)

In [ ]:
plt.plot(n_donnees_entrainement, dnn_pretrain_test_error, label="DNN pré-entraîné (test)")
plt.scatter(n_donnees_entrainement, dnn_pretrain_test_error)
plt.plot(n_donnees_entrainement, dnn_random_test_error, label="DNN non pré-entraîné (test)")
plt.scatter(n_donnees_entrainement, dnn_random_test_error)
plt.xlabel("nombre de données d'entraînement")
plt.ylabel("taux d'erreur")
plt.legend()
plt.show()

print("Erreurs train (pré-entraîné):", np.round(dnn_pretrain_train_error, 4))
print("Erreurs test  (pré-entraîné):", np.round(dnn_pretrain_test_error, 4))
print("Erreurs train (aléatoire):", np.round(dnn_random_train_error, 4))
print("Erreurs test  (aléatoire):", np.round(dnn_random_test_error, 4))

### 4) Vérification des probabilités de sortie
Observation des probabilités softmax sur quelques images du jeu d'apprentissage (après entraînement).

In [ ]:
np.random.seed(42)
indices = np.random.choice(len(train_data), size=5, replace=False)
x_samples = train_data[indices]
y_true = train_clear_label[indices]

# Utilise le dernier modèle pré-entraîné appris (variable dnn_pretrain)
_, probs = dnn_pretrain.entree_sortie_reseau(x_samples)
y_pred = np.argmax(probs, axis=1)

print("Indices sélectionnés:", indices)
print("Labels réels      :", y_true)
print("Prédictions       :", y_pred)
print("Probabilités softmax (5 exemples x 10 classes):")
print(np.round(probs, 4))

In [ ]:
# Entraînement + comparaison des probabilités de sortie sur plusieurs images
couches_compare = [n_pixels, 200, 200]
n_train_compare = 10000
n_samples_compare = 8

# 1) Modèle avec pré-entraînement
dnn_pretrain_compare = DNN()
dnn_pretrain_compare.init_DNN(couches_compare, classification_size)
dnn_pretrain_compare.pretrain_DNN(train_data[:n_train_compare], num_iterations, learning_rate, batch_size)
dnn_pretrain_compare.retropropagation(
    num_iterations_mnist, learning_rate, batch_size,
    train_data[:n_train_compare], train_labels[:n_train_compare]
 )

# 2) Modèle sans pré-entraînement
dnn_random_compare = DNN()
dnn_random_compare.init_DNN(couches_compare, classification_size)
dnn_random_compare.retropropagation(
    num_iterations_mnist, learning_rate, batch_size,
    train_data[:n_train_compare], train_labels[:n_train_compare]
 )

# 3) Sélection de plusieurs images de test
np.random.seed(7)
idx = np.random.choice(len(test_data), size=n_samples_compare, replace=False)
x_cmp = test_data[idx]
y_cmp = test_clear_label[idx]

# 4) Probabilités de sortie
_, probs_pre = dnn_pretrain_compare.entree_sortie_reseau(x_cmp)
_, probs_rand = dnn_random_compare.entree_sortie_reseau(x_cmp)

pred_pre = np.argmax(probs_pre, axis=1)
pred_rand = np.argmax(probs_rand, axis=1)

print("Indices test:", idx)
print("Labels réels:", y_cmp)
print("Prédictions (pré-entraîné):", pred_pre)
print("Prédictions (sans pré-entraînement):", pred_rand)

# 5) Visualisation des images sélectionnées
n_cols_show = 4
n_rows_show = int(np.ceil(n_samples_compare / n_cols_show))
fig, axes = plt.subplots(n_rows_show, n_cols_show, figsize=(3 * n_cols_show, 3 * n_rows_show))
axes = np.array(axes).reshape(-1)

for i in range(n_samples_compare):
    img = x_cmp[i].reshape(n_rows, n_cols)
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f"y={y_cmp[i]} | pre={pred_pre[i]} | rnd={pred_rand[i]}")
    axes[i].axis('off')

for j in range(n_samples_compare, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

# 6) Visualisation comparative des probabilités (classes 0..9)
y_positions = np.arange(n_samples_compare)
y_labels = [str(int(label)) for label in y_cmp]

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
im0 = axes[0].imshow(probs_pre, aspect='auto', cmap='Blues')
axes[0].set_title("Probabilités softmax - DNN pré-entraîné")
axes[0].set_ylabel("Label réel")
axes[0].set_yticks(y_positions)
axes[0].set_yticklabels(y_labels)
plt.colorbar(im0, ax=axes[0], fraction=0.02, pad=0.02)

im1 = axes[1].imshow(probs_rand, aspect='auto', cmap='Oranges')
axes[1].set_title("Probabilités softmax - DNN sans pré-entraînement")
axes[1].set_xlabel("Classe")
axes[1].set_ylabel("Label réel")
axes[1].set_yticks(y_positions)
axes[1].set_yticklabels(y_labels)
axes[1].set_xticks(range(classification_size))
plt.colorbar(im1, ax=axes[1], fraction=0.02, pad=0.02)

plt.tight_layout()
plt.show()